<a href="https://colab.research.google.com/github/diyayourfav/AI-MLModule/blob/main/DiyaBudhathoki_Workshop6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Ai_Ml/Data/Week5/FruitinAmazon.zip"
extract_path = "/content/FruitinAmazon/FruitinAmazon"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
os.listdir("/content/FruitinAmazon/FruitinAmazon/test")

['graviola', 'guarana', 'acai', 'pupunha', 'cupuacu', 'tucuma']

In [22]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    "/content/FruitinAmazon/FruitinAmazon/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 72 images belonging to 6 classes.
Found 18 images belonging to 6 classes.
Found 30 images belonging to 6 classes.


In [23]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [15]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.1111 - loss: 2.5908 - val_accuracy: 0.3889 - val_loss: 1.4888
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 930ms/step - accuracy: 0.3889 - loss: 1.4937 - val_accuracy: 0.6667 - val_loss: 1.1101
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 933ms/step - accuracy: 0.7222 - loss: 0.8874 - val_accuracy: 0.7778 - val_loss: 0.8374
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 637ms/step - accuracy: 0.7222 - loss: 0.7886 - val_accuracy: 0.8333 - val_loss: 0.6261
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 668ms/step - accuracy: 0.8889 - loss: 0.4133 - val_accuracy: 0.8889 - val_loss: 0.5067
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 753ms/step - accuracy: 0.9444 - loss: 0.2487 - val_accuracy: 0.8889 - val_loss: 0.4622
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 920ms/step - accuracy: 0.9583 - loss: 0.2467 - val_accuracy: 0.9444 - val_loss: 0.4364
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 653ms/step - accuracy: 0.9167 - loss: 0.2473 - val_accuracy: 0.9444 - val_loss: 0.

In [16]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
              precision    recall  f1-score   support

        acai       0.80      0.80      0.80         5
     cupuacu       1.00      1.00      1.00         5
    graviola       1.00      1.00      1.00         5
     guarana       0.83      1.00      0.91         5
     pupunha       0.62      1.00      0.77         5
      tucuma       1.00      0.20      0.33         5

    accuracy                           0.83        30
   macro avg       0.88      0.83      0.80        30
weighted avg       0.88      0.83      0.80        30



In [17]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/FruitinAmazon/FruitinAmazon/train/pupunha/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 901ms/step
Prediction: pupunha


In [18]:
from tensorflow.keras import Sequential

scratch_model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_data, validation_data=val_data, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.1250 - loss: 9.2917 - val_accuracy: 0.1667 - val_loss: 8.8097
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.1944 - loss: 5.3778 - val_accuracy: 0.2222 - val_loss: 3.9177
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.3056 - loss: 2.1493 - val_accuracy: 0.3889 - val_loss: 1.7377
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.5694 - loss: 1.3342 - val_accuracy: 0.3889 - val_loss: 1.6065
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.6806 - loss: 1.0170 - val_accuracy: 0.3889 - val_loss: 1.7095
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.8333 - loss: 0.5846 - val_accuracy: 0.5556 - val_loss: 1.4615
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.9167 - loss: 0.3983 - val_accuracy: 0.5000 - val_loss: 1.8936
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.8750 - loss: 0.3015 - val_accuracy: 0.3889 - val_loss: 1.2230
Epoch 9/10
3/3 ━

In [24]:
predictions = scratch_model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 493ms/step
              precision    recall  f1-score   support

        acai       1.00      0.60      0.75         5
     cupuacu       0.25      0.60      0.35         5
    graviola       0.57      0.80      0.67         5
     guarana       0.80      0.80      0.80         5
     pupunha       0.67      0.40      0.50         5
      tucuma       0.00      0.00      0.00         5

    accuracy                           0.53        30
   macro avg       0.55      0.53      0.51        30
weighted avg       0.55      0.53      0.51        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/FruitinAmazon/FruitinAmazon/train/pupunha/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = scratch_model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Prediction: pupunha
